# Notebook 05 — Seleção dos modelos de sentimento para a predição da inadimplência

## Objetivo do notebook

Este notebook tem como objetivo selecionar quais modelos de análise de sentimento
serão empregados no modelo de predição da inadimplência (Notebook 06).

A lógica aplicada é:

1. carregar a série mensal de inadimplência, vinda do Notebook 02;
2. carregar as séries de sentimento geradas no Notebook 04 para os modelos
   BERT, Mistral, TextBlob, NLTK/VADER e FinBERT;
3. criar um sexto modelo, chamado **Média dos Modelos**, calculado como a média
   dos cinco scores de sentimento;
4. construir uma série mensal de sentimento com lags de 1 a 6 meses e fazer o
   merge com a base de inadimplência — essa base é exportada para o Notebook 06;
5. calcular a correlação entre o sentimento do mês `t` e a inadimplência futura
   `inad_total_tplus3` (inadimplência em `t+3`), sem defasagem adicional do
   sentimento, usando a série completa;
6. selecionar os quatro modelos com maior correlação absoluta para uso no
   Notebook 06.

> **Sobre as variáveis-alvo:** `inad_total_tplus3` é a inadimplência observada
> 3 meses à frente do mês de referência `t`, e `data_alvo` é o mês calendário
> correspondente a essa observação (`data_alvo = t + 3 meses`). Ambas as colunas
> foram criadas no Notebook 02, onde o horizonte de previsão H=3 foi definido.
>
> **Sobre os lags de sentimento:** os lags (L1 a L6) são construídos neste
> notebook e exportados para o Notebook 06, onde o modelo aprenderá quais
> defasagens são mais preditivas. A correlação da seção 6, usada apenas para
> selecionar os modelos, é calculada sem lag adicional.

In [48]:
!pip install seaborn


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## 1. Carregamento das bases

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Série de Inadimplência (Notebook 02) 
df_inad = pd.read_csv("base_modelagem_inadimplencia.csv")



# ── Scores de Sentimento (Notebook 04_completo) 
URL_SENT = "https://raw.githubusercontent.com/akemitti/Pred-inad-credito/main/base_sentimentos.csv"
try:
    df_sent = pd.read_csv("base_sentimentos.csv")
    print("base_sentimentos.csv carregado do disco.")
except FileNotFoundError:
    df_sent = pd.read_csv(URL_SENT)
    print("base_sentimentos.csv carregado do GitHub.")

df_sent.columns = [c.strip().lower() for c in df_sent.columns]
df_sent["data"] = pd.to_datetime(df_sent["data"], errors="coerce")

# Identificar colunas de score
score_cols_raw = [c for c in df_sent.columns if c.startswith("score_")]
print(f"Scores disponíveis: {score_cols_raw}")


base_sentimentos.csv carregado do disco.
Scores disponíveis: ['score_nltk', 'score_textblob', 'score_bert', 'score_finbert', 'score_mistral']


In [50]:
df_inad

,Unnamed: 0,data,data_alvo,inad_total_tplus3,inad_total,inad_total_L1,inad_total_L2,inad_total_L3,inad_total_L4,inad_total_L5,inad_total_L6
0,6,2019-07-01,2019-10-01,3.03,3.06,2.95,3.05,3.02,2.99,2.91,2.95
1,7,2019-08-01,2019-11-01,3.00,3.04,3.06,2.95,3.05,3.02,2.99,2.91
2,8,2019-09-01,2019-12-01,2.94,3.06,3.04,3.06,2.95,3.05,3.02,2.99
3,9,2019-10-01,2020-01-01,3.00,3.03,3.06,3.04,3.06,2.95,3.05,3.02
4,10,2019-11-01,2020-02-01,3.04,3.00,3.03,3.06,3.04,3.06,2.95,3.05
...,...,...,...,...,...,...,...,...,...,...,...
70,76,2025-05-01,2025-08-01,3.95,3.54,3.50,3.28,3.26,3.19,2.95,3.14
71,77,2025-06-01,2025-09-01,3.91,3.57,3.54,3.50,3.28,3.26,3.19,2.95
72,78,2025-07-01,2025-10-01,4.00,3.77,3.57,3.54,3.50,3.28,3.26,3.19
73,79,2025-08-01,2025-11-01,4.05,3.95,3.77,3.57,3.54,3.50,3.28,3.26


In [51]:
df_sent

,data,tipo_relatorio,arquivo,tipo,score_nltk,score_textblob,score_bert,score_finbert,score_mistral
0,2019-02-06,copom,COPOM220-not20190206220.pdf,copom,-0.7783,0.000000,-0.071429,-0.248734,0.50
1,2019-03-20,copom,COPOM221-not20190320221.pdf,copom,-0.6808,0.000000,-0.062500,-0.336417,0.65
2,2019-05-08,copom,COPOM222-not20190508222.pdf,copom,-0.9274,0.062500,-0.125000,-0.388865,0.65
3,2019-06-19,copom,Copom223-not20190619223.pdf,copom,-0.7783,0.000000,-0.187500,-0.407422,0.65
4,2019-07-31,copom,Copom224-not20190731224.pdf,copom,-0.8402,0.000000,0.062500,-0.227865,0.75
...,...,...,...,...,...,...,...,...,...
129,2025-08-01,estatisticas,202508_Texto_de_estatisticas_monetarias_e_de_c...,estatisticas,-0.9894,0.000000,-0.916667,-0.018209,-0.10
130,2025-09-01,estatisticas,202509_Texto_de_estatisticas_monetarias_e_de_c...,estatisticas,-0.9776,0.050000,-1.000000,0.007749,-0.60
131,2025-10-01,estatisticas,202510_Texto_de_estatisticas_monetarias_e_de_c...,estatisticas,-0.9920,-0.400000,-1.000000,-0.007200,0.65
132,2025-11-01,estatisticas,202511_Texto_de_estatisticas_monetarias_e_de_c...,estatisticas,-0.9898,-0.266667,-1.000000,0.292047,-0.30


## 2. Agregação mensal com grade completa e interpolação

A série de inadimplência é mensal. Como as **Atas do Copom** não ocorrem exatamente todos os meses — em geral, seguem o calendário de reuniões do Copom, que pode ocorrer em intervalos próximos de 45 dias — é necessário transformar o sentimento das atas em uma série mensal contínua.

A regra aplicada neste notebook é:

1. agregar os scores de sentimento por mês e por tipo de relatório;
2. criar uma grade mensal completa de `2019-01-01` até `2025-12-01`;
3. garantir que, para todos os meses, exista valor de sentimento para as **Atas do Copom**, usando interpolação linear para preencher lacunas internas;
4. aplicar `bfill/ffill` — preenchimento com o valor seguinte/anterior — apenas nas bordas da série, quando necessário;
5. manter colunas de rastreabilidade para identificar quais meses eram observados originalmente e quais foram preenchidos.

Os Relatórios de Estatísticas também são mantidos na base consolidada, mas o ponto metodológico mais importante aqui é a mensalização das Atas do Copom para compatibilização com a série mensal de inadimplência.

In [52]:
df_sent_completo = df_sent.copy()

# ── Padronização da identificação do tipo de relatório ──────────────────────
if "tipo_relatorio" not in df_sent_completo.columns:
    if "tipo" in df_sent_completo.columns:
        df_sent_completo["tipo_relatorio"] = df_sent_completo["tipo"]
    else:
        raise ValueError(
            "A coluna 'tipo_relatorio' não foi encontrada na base_sentimentos.csv. "
            "Verifique se o Notebook 04 exportou a base consolidada corretamente."
        )

df_sent_completo["tipo_relatorio"] = (
    df_sent_completo["tipo_relatorio"]
    .astype(str)
    .str.lower()
    .str.strip()
)

TIPOS_VALIDOS = ["copom", "estatisticas"]
df_sent_completo = df_sent_completo[
    df_sent_completo["tipo_relatorio"].isin(TIPOS_VALIDOS)
].copy()

# ── Padronização da data e dos scores ───────────────────────────────────────
df_sent_completo["data"] = pd.to_datetime(df_sent_completo["data"], errors="coerce")
df_sent_completo = df_sent_completo.dropna(subset=["data"])

# Garantir que os scores sejam numéricos
for col in score_cols_raw:
    df_sent_completo[col] = pd.to_numeric(df_sent_completo[col], errors="coerce")

score_cols = [c for c in score_cols_raw if df_sent_completo[c].notna().any()]

print("Scores válidos para análise:")
print(score_cols)

print("\nDistribuição da base completa por tipo de relatório:")
display(
    df_sent_completo["tipo_relatorio"]
    .value_counts()
    .rename_axis("tipo_relatorio")
    .reset_index(name="qtd_registros")
)

# ── Grade mensal completa ───────────────────────────────────────────────────
PERIODO_INICIO = "2019-01-01"
PERIODO_FIM = "2025-12-01"

grade_mensal = pd.date_range(
    start=PERIODO_INICIO,
    end=PERIODO_FIM,
    freq="MS"
)

print(f"\nGrade mensal esperada: {len(grade_mensal)} meses")
print(f"Período: {grade_mensal.min().date()} a {grade_mensal.max().date()}")


def agregar_mensal_com_interpolacao(df_base, tipo_relatorio, score_cols, grade_mensal):
    """
    Agrega os scores por mês para um tipo de relatório e garante uma série mensal completa.

    Para as Atas do Copom, essa etapa é essencial porque as atas não são mensais.
    A interpolação linear preenche lacunas internas entre observações conhecidas.
    """
    df_tipo = df_base[df_base["tipo_relatorio"] == tipo_relatorio].copy()

    if df_tipo.empty:
        raise ValueError(f"Não há registros para tipo_relatorio = '{tipo_relatorio}'.")

    df_tipo["mes"] = df_tipo["data"].dt.to_period("M").dt.to_timestamp()

    # Agregação mensal: se houver mais de um documento no mesmo mês, usa a média.
    df_obs = (
        df_tipo.groupby("mes")[score_cols]
        .mean()
        .sort_index()
    )

    meses_observados = set(df_obs.index)

    # Grade mensal completa
    df_mensal = df_obs.reindex(grade_mensal)
    df_mensal.index.name = "data"
    df_mensal = df_mensal.reset_index()

    df_mensal["tipo_relatorio"] = tipo_relatorio
    df_mensal["registro_original"] = df_mensal["data"].isin(meses_observados)
    df_mensal["mes_ausente_originalmente"] = ~df_mensal["registro_original"]

    # Preenchimento das lacunas internas
    scores_interpolados = df_mensal[score_cols].interpolate(
        method="linear",
        limit_area="inside"
    )

    # Preenchimento de bordas, quando necessário
    scores_preenchidos = scores_interpolados.bfill().ffill()

    df_mensal[score_cols] = scores_preenchidos

    # Rastreabilidade do método de preenchimento
    df_mensal["metodo_preenchimento"] = "observado"

    mask_interpolado = (
        df_mensal["mes_ausente_originalmente"]
        & scores_interpolados.notna().any(axis=1)
    )
    df_mensal.loc[mask_interpolado, "metodo_preenchimento"] = "interpolacao_linear"

    mask_borda = (
        df_mensal["mes_ausente_originalmente"]
        & ~mask_interpolado
    )
    df_mensal.loc[mask_borda, "metodo_preenchimento"] = "bfill_ffill_borda"

    return df_mensal


# Gerar base mensal completa para cada tipo de relatório
dfs_mensais = []
for tipo in TIPOS_VALIDOS:
    df_tipo_mensal = agregar_mensal_com_interpolacao(
        df_base=df_sent_completo,
        tipo_relatorio=tipo,
        score_cols=score_cols,
        grade_mensal=grade_mensal
    )
    dfs_mensais.append(df_tipo_mensal)

df_sent_mensal = pd.concat(dfs_mensais, ignore_index=True)

# Verificação da cobertura mensal
resumo_cobertura = (
    df_sent_mensal
    .groupby("tipo_relatorio")
    .agg(
        meses_total=("data", "nunique"),
        registros_originais=("registro_original", "sum"),
        meses_preenchidos=("mes_ausente_originalmente", "sum")
    )
    .reset_index()
)

print("\n=== Cobertura mensal após interpolação ===")
display(resumo_cobertura)

print("\nMétodo de preenchimento por tipo de relatório:")
display(
    df_sent_mensal
    .groupby(["tipo_relatorio", "metodo_preenchimento"])
    .size()
    .reset_index(name="qtd_meses")
)

# Checagem específica: Copom deve ter todos os meses.
qtd_meses_copom = df_sent_mensal.query("tipo_relatorio == 'copom'")["data"].nunique()
assert qtd_meses_copom == len(grade_mensal), (
    f"Copom deveria ter {len(grade_mensal)} meses, mas tem {qtd_meses_copom}."
)

print(f"\n✅ Série mensal do Copom completa: {qtd_meses_copom} meses.")
display(df_sent_mensal.head())

Scores válidos para análise:
['score_nltk', 'score_textblob', 'score_bert', 'score_finbert', 'score_mistral']

Distribuição da base completa por tipo de relatório:


,tipo_relatorio,qtd_registros
0,estatisticas,81
1,copom,53



Grade mensal esperada: 84 meses
Período: 2019-01-01 a 2025-12-01

=== Cobertura mensal após interpolação ===


,tipo_relatorio,meses_total,registros_originais,meses_preenchidos
0,copom,84,53,31
1,estatisticas,84,81,3



Método de preenchimento por tipo de relatório:


,tipo_relatorio,metodo_preenchimento,qtd_meses
0,copom,bfill_ffill_borda,1
1,copom,interpolacao_linear,30
2,copom,observado,53
3,estatisticas,interpolacao_linear,3
4,estatisticas,observado,81



✅ Série mensal do Copom completa: 84 meses.


,data,score_nltk,score_textblob,score_bert,score_finbert,score_mistral,tipo_relatorio,registro_original,mes_ausente_originalmente,metodo_preenchimento
0,2019-01-01,-0.7783,0.00000,-0.071429,-0.248734,0.50,copom,False,True,bfill_ffill_borda
1,2019-02-01,-0.7783,0.00000,-0.071429,-0.248734,0.50,copom,True,False,observado
2,2019-03-01,-0.6808,0.00000,-0.062500,-0.336417,0.65,copom,True,False,observado
3,2019-04-01,-0.8041,0.03125,-0.093750,-0.362641,0.65,copom,False,True,interpolacao_linear
4,2019-05-01,-0.9274,0.06250,-0.125000,-0.388865,0.65,copom,True,False,observado


## 3. Modelo 6 — Média dos modelos de sentimento

In [53]:
# Sentimento médio dos modelos válidos
df_sent_mensal["score_media"] = df_sent_mensal[score_cols].mean(axis=1)
score_cols_todos = score_cols + ["score_media"]

print("Score médio (primeiros 5):")
df_sent_mensal



Score médio (primeiros 5):


,data,score_nltk,score_textblob,score_bert,score_finbert,score_mistral,tipo_relatorio,registro_original,mes_ausente_originalmente,metodo_preenchimento,score_media
0,2019-01-01,-0.7783,0.000000,-0.071429,-0.248734,0.50,copom,False,True,bfill_ffill_borda,-0.119692
1,2019-02-01,-0.7783,0.000000,-0.071429,-0.248734,0.50,copom,True,False,observado,-0.119692
2,2019-03-01,-0.6808,0.000000,-0.062500,-0.336417,0.65,copom,True,False,observado,-0.085943
3,2019-04-01,-0.8041,0.031250,-0.093750,-0.362641,0.65,copom,False,True,interpolacao_linear,-0.115848
4,2019-05-01,-0.9274,0.062500,-0.125000,-0.388865,0.65,copom,True,False,observado,-0.145753
...,...,...,...,...,...,...,...,...,...,...,...
163,2025-08-01,-0.9894,0.000000,-0.916667,-0.018209,-0.10,estatisticas,True,False,observado,-0.404855
164,2025-09-01,-0.9776,0.050000,-1.000000,0.007749,-0.60,estatisticas,True,False,observado,-0.503970
165,2025-10-01,-0.9920,-0.400000,-1.000000,-0.007200,0.65,estatisticas,True,False,observado,-0.349840
166,2025-11-01,-0.9898,-0.266667,-1.000000,0.292047,-0.30,estatisticas,True,False,observado,-0.452884


## 4. Preparação das bases para geração dos lags e merge

Nesta etapa, a base de inadimplência é apenas preparada para o merge, mas **ainda não é unida** à base de sentimento.

A lógica adotada é:

1. a base mensal de sentimento começa em `2019-01-01`;
2. os lags dos scores de sentimento são calculados sobre essa série mensal completa;
3. depois dos lags, a base de sentimento é limpa com `dropna()`;
4. somente depois dessa limpeza é feito o merge com a base de inadimplência.

Esse fluxo evita perder a informação de janeiro a junho de 2019 no cálculo dos lags. Por exemplo, o lag 6 de julho de 2019 usa o sentimento de janeiro de 2019.



In [54]:
# Cópias para evitar alterar os dataframes originais diretamente
# dataframe = estrutura tabular do pandas, parecida com uma planilha
df_inad_merge = df_inad.copy()
df_sent_lag_base = df_sent_mensal.copy()

# ── Parâmetro principal da etapa ────────────────────────────────────────────
# Os lags de sentimento devem ser calculados a partir de 2019-01-01,
# antes do merge com a base de inadimplência.
DATA_INICIO_LAGS_SENTIMENTO = pd.Timestamp("2019-01-01")

# ── Preparação da base de inadimplência ─────────────────────────────────────
# A base de inadimplência já vem do Notebook 02 com os lags da própria série,
# como inad_total_L1, inad_total_L2, etc. Esses lags NÃO serão recriados aqui.
df_inad_merge.columns = [c.strip().lower() for c in df_inad_merge.columns]
df_inad_merge = df_inad_merge.drop(columns=["unnamed: 0", "Unnamed: 0"], errors="ignore")

if "data" not in df_inad_merge.columns:
    raise ValueError("A base de inadimplência precisa ter uma coluna chamada 'data'.")

df_inad_merge["data"] = pd.to_datetime(df_inad_merge["data"], errors="coerce")
df_inad_merge["mes"] = df_inad_merge["data"].dt.to_period("M").dt.to_timestamp()

df_inad_merge = (
    df_inad_merge
    .dropna(subset=["mes"])
    .drop_duplicates(subset=["mes"])
    .sort_values("mes")
    .reset_index(drop=True)
)

# ── Preparação da base mensal de sentimento ─────────────────────────────────
df_sent_lag_base["data"] = pd.to_datetime(df_sent_lag_base["data"], errors="coerce")
df_sent_lag_base["mes"] = df_sent_lag_base["data"].dt.to_period("M").dt.to_timestamp()

df_sent_lag_base = (
    df_sent_lag_base
    .dropna(subset=["mes", "tipo_relatorio"])
    .query("mes >= @DATA_INICIO_LAGS_SENTIMENTO")
    .sort_values(["tipo_relatorio", "mes"])
    .reset_index(drop=True)
)

# Conferir se a base mensal de sentimento tem apenas uma linha por mês e tipo de relatório.
# Isso é importante porque os lags precisam ser calculados em uma série temporal mensal única.
duplicados_sent = df_sent_lag_base[
    df_sent_lag_base.duplicated(subset=["mes", "tipo_relatorio"], keep=False)
]

if not duplicados_sent.empty:
    print("Atenção: foram encontrados meses duplicados na base mensal de sentimento.")
    display(duplicados_sent[["data", "mes", "tipo_relatorio"] + score_cols_todos].head(10))
else:
    print("✅ Base de sentimento mensal sem duplicidade por mês e tipo de relatório.")

print("=== Bases preparadas ===")
print(f"Inadimplência: {len(df_inad_merge)} linhas | período: {df_inad_merge['mes'].min().date()} a {df_inad_merge['mes'].max().date()}")
print(f"Sentimento mensal para lags: {len(df_sent_lag_base)} linhas | período: {df_sent_lag_base['mes'].min().date()} a {df_sent_lag_base['mes'].max().date()}")

print("\nCobertura da base de sentimento por tipo de relatório:")
display(
    df_sent_lag_base
    .groupby("tipo_relatorio")
    .agg(
        inicio=("mes", "min"),
        fim=("mes", "max"),
        qtd_meses=("mes", "nunique")
    )
    .reset_index()
)



✅ Base de sentimento mensal sem duplicidade por mês e tipo de relatório.
=== Bases preparadas ===
Inadimplência: 75 linhas | período: 2019-07-01 a 2025-09-01
Sentimento mensal para lags: 168 linhas | período: 2019-01-01 a 2025-12-01

Cobertura da base de sentimento por tipo de relatório:


,tipo_relatorio,inicio,fim,qtd_meses
0,copom,2019-01-01,2025-12-01,84
1,estatisticas,2019-01-01,2025-12-01,84


## 5. Geração dos lags de sentimento antes do merge

A base final exportada para o Notebook 06 deve conter todos os modelos de sentimento com defasagens de 1 até 6 meses.

A convenção adotada é:

- `_L1`: sentimento defasado em 1 mês;
- `_L2`: sentimento defasado em 2 meses;
- `_L3`: sentimento defasado em 3 meses;
- ...
- `_L6`: sentimento defasado em 6 meses.

Exemplo de notação:

```text
copom_score_mistral_L1
copom_score_mistral_L2
copom_score_mistral_L3
copom_score_mistral_L4
copom_score_mistral_L5
copom_score_mistral_L6
```

Importante: os lags são calculados sobre a série mensal de sentimento iniciada em `2019-01-01`. Depois disso, a base de sentimento com lags é limpa com `dropna()`. Só então ocorre o merge com a base de inadimplência.

Importante: A coluna data_alvo vem do Notebook 02 e equivale a data + 3 meses, pois o modelo prevê a inadimplência com horizonte h+3. Assim, a relação modelada é: sentimento em (data − lag) → inadimplência em (data + 3), com lead total = lag + 3 meses.

In [55]:
## 6. Análise de correlação

# Lag fixo justificado: a transmissão da política monetária para o crédito
# leva tipicamente 2–4 meses no Brasil. Usamos L3 como ponto central desse
# intervalo. Fixar o lag garante que todos os modelos sejam comparados
# nas mesmas condições — sem cherry-picking do melhor lag por modelo.

LAG_FIXO = 3
COLUNA_INAD = "inad_total_tplus3"

if COLUNA_INAD not in df_export_final.columns:
    raise ValueError(f"Coluna {COLUNA_INAD} não encontrada. Verifique o Notebook 02.")

nomes_display = {
    "score_nltk":     "NLTK/VADER",
    "score_textblob": "TextBlob",
    "score_bert":     "BERT Multilingual",
    "score_finbert":  "FinBERT-PT-BR",
    "score_mistral":  "Mistral",
    "score_media":    "Média dos Modelos",
}

resultados = []

for tipo in TIPOS_VALIDOS:
    for col in score_cols_todos:
        col_lag = f"{tipo}_{col}_L{LAG_FIXO}"

        if col_lag not in df_export_final.columns:
            continue

        # Série completa — sem filtro de período de treino.
        # O objetivo aqui é selecionar o modelo, não avaliar previsão.
        df_tmp = (
            df_export_final[[COLUNA_INAD, col_lag]]
            .dropna()
        )

        if len(df_tmp) < 5:
            continue
        if df_tmp[COLUNA_INAD].nunique() <= 1 or df_tmp[col_lag].nunique() <= 1:
            continue

        r_p,  p_p  = stats.pearsonr(df_tmp[COLUNA_INAD],  df_tmp[col_lag])
        r_s,  p_s  = stats.spearmanr(df_tmp[COLUNA_INAD], df_tmp[col_lag])

        resultados.append({
            "Tipo Relatório": tipo,
            "Modelo":         nomes_display.get(col, col),
            "col":            col,
            "col_lag":        col_lag,
            "Pearson r":      round(r_p, 4),
            "Pearson p":      round(p_p, 4),
            "Spearman ρ":     round(r_s, 4),
            "Spearman p":     round(p_s, 4),
            "|Pearson r|":    round(abs(r_p), 4),
            "N observações":  len(df_tmp),
        })

df_corr = pd.DataFrame(resultados)

print(f"=== Correlação: sentimento (L{LAG_FIXO}) × inadimplência (t+3) — série completa ===")
display(
    df_corr
    .sort_values(["|Pearson r|"], ascending=False)
    .drop(columns=["col", "col_lag"])
    .reset_index(drop=True)
)

=== Correlação: sentimento (L3) × inadimplência (t+3) — série completa ===


,Tipo Relatório,Modelo,Pearson r,Pearson p,Spearman ρ,Spearman p,|Pearson r|,N observações
0,copom,Mistral,-0.4931,0.0000,-0.5142,0.0000,0.4931,75
1,copom,NLTK/VADER,0.3617,0.0014,0.4015,0.0004,0.3617,75
2,copom,Média dos Modelos,-0.3588,0.0016,-0.3144,0.0060,0.3588,75
3,estatisticas,TextBlob,0.2781,0.0157,0.3038,0.0080,0.2781,75
4,copom,BERT Multilingual,0.2708,0.0188,0.2500,0.0305,0.2708,75
5,copom,TextBlob,-0.1772,0.1284,0.0667,0.5699,0.1772,75
6,estatisticas,FinBERT-PT-BR,-0.1296,0.2679,-0.1277,0.2751,0.1296,75
7,estatisticas,Mistral,-0.1251,0.2848,-0.1094,0.3501,0.1251,75
8,copom,FinBERT-PT-BR,0.1022,0.3828,0.0196,0.8676,0.1022,75
9,estatisticas,NLTK/VADER,-0.0810,0.4895,-0.0639,0.5859,0.0810,75


In [56]:
# Exportação principal para consumo no Notebook 06
# index=False evita criar uma coluna extra chamada "Unnamed: 0".
df_export_final.to_csv("base_completa.csv", index=False)

print("Arquivo exportado: base_completa.csv")


Arquivo exportado: base_completa.csv


## 6. Análise de correlação

Para selecionar quais modelos de sentimento entrarão no Notebook 06, calculamos
a correlação entre o **sentimento observado no mês `t`** e a
**inadimplência em `t+3`** (`inad_total_tplus3`).

Não é aplicado nenhum lag adicional ao sentimento nesta etapa. A variável-alvo
`inad_total_tplus3` já representa a inadimplência futura, de modo que a
correlação medida capta diretamente a associação entre o tom dos documentos hoje
e o comportamento do crédito nos três meses seguintes.

A correlação é calculada sobre a **série completa** (2019–2025), sem separação
entre treino e teste. O objetivo aqui é ranquear os modelos entre si em condições
iguais — não avaliar capacidade preditiva, o que será feito no Notebook 06.

São reportadas as correlações de **Pearson** (associação linear) e
**Spearman** (associação monotônica), e o critério de ordenação é o valor
absoluto do coeficiente de Pearson.


In [ ]:
# Correlação entre sentimento do mês t e inadimplência em t+3.
# Não é aplicado lag adicional ao sentimento: a variável inad_total_tplus3
# já representa a inadimplência 3 meses à frente do mês de referência,
# de forma que a associação medida é direta — sentimento hoje vs. inadimplência futura.

COLUNA_INAD = "inad_total_tplus3"

if COLUNA_INAD not in df_export_final.columns:
    raise ValueError(f"Coluna {COLUNA_INAD} não encontrada. Verifique o Notebook 02.")

nomes_display = {
    "score_nltk":     "NLTK/VADER",
    "score_textblob": "TextBlob",
    "score_bert":     "BERT Multilingual",
    "score_finbert":  "FinBERT-PT-BR",
    "score_mistral":  "Mistral",
    "score_media":    "Média dos Modelos",
}

# df_export_final contém apenas lags L1...L6, não o sentimento bruto.
# Buscamos os scores mensais sem defasagem direto de df_sent_lag_base.
df_inad_target = (
    df_export_final[["data", COLUNA_INAD]]
    .rename(columns={"data": "mes"})
    .copy()
)

resultados = []

for tipo in TIPOS_VALIDOS:
    df_tipo_raw = (
        df_sent_lag_base[df_sent_lag_base["tipo_relatorio"] == tipo]
        [["mes"] + score_cols_todos]
        .copy()
    )

    df_merged = df_tipo_raw.merge(df_inad_target, on="mes", how="inner")

    for col in score_cols_todos:
        df_tmp = df_merged[[COLUNA_INAD, col]].dropna()

        if len(df_tmp) < 5:
            continue
        if df_tmp[COLUNA_INAD].nunique() <= 1 or df_tmp[col].nunique() <= 1:
            continue

        r_p, p_p = stats.pearsonr(df_tmp[COLUNA_INAD],  df_tmp[col])
        r_s, p_s = stats.spearmanr(df_tmp[COLUNA_INAD], df_tmp[col])

        resultados.append({
            "Tipo Relatório": tipo,
            "Modelo":         nomes_display.get(col, col),
            "col":            col,
            "Pearson r":      round(r_p, 4),
            "Pearson p":      round(p_p, 4),
            "Spearman ρ":     round(r_s, 4),
            "Spearman p":     round(p_s, 4),
            "|Pearson r|":    round(abs(r_p), 4),
            "N observações":  len(df_tmp),
        })

df_corr = pd.DataFrame(resultados)

print("=== Correlação: sentimento (mês t) × inadimplência (t+3) — série completa ===")
display(
    df_corr
    .sort_values("|Pearson r|", ascending=False)
    .reset_index(drop=True)
)

=== Correlação: sentimento (mês t) × inadimplência (t+3) — série completa ===


,Tipo Relatório,Modelo,col,Pearson r,Pearson p,Spearman ρ,Spearman p,|Pearson r|,N observações
0,copom,NLTK/VADER,score_nltk,0.4292,0.0001,0.4377,0.0001,0.4292,75
1,copom,Mistral,score_mistral,-0.4218,0.0002,-0.4099,0.0003,0.4218,75
2,copom,Média dos Modelos,score_media,-0.3029,0.0083,-0.2742,0.0173,0.3029,75
3,estatisticas,FinBERT-PT-BR,score_finbert,-0.2733,0.0177,-0.3201,0.0051,0.2733,75
4,estatisticas,TextBlob,score_textblob,0.1739,0.1356,0.1374,0.2398,0.1739,75
5,copom,BERT Multilingual,score_bert,0.1706,0.1433,0.1776,0.1274,0.1706,75
6,estatisticas,Média dos Modelos,score_media,-0.0873,0.4563,-0.1425,0.2226,0.0873,75
7,copom,FinBERT-PT-BR,score_finbert,0.0627,0.5928,0.0502,0.6690,0.0627,75
8,copom,TextBlob,score_textblob,-0.0604,0.6067,0.1388,0.2350,0.0604,75
9,estatisticas,Mistral,score_mistral,-0.0438,0.7090,-0.0425,0.7173,0.0438,75


## 7. Ranking diagnóstico das correlações

A tabela abaixo organiza os modelos em ordem decrescente de correlação absoluta
com a inadimplência futura (`inad_total_tplus3`).

Os quatro modelos com maior `|Pearson r|` serão levados para o Notebook 06.
A escolha final — incluindo o número de lags a usar — será validada lá pelo
desempenho preditivo (RMSE em walk-forward), não pela correlação.

In [58]:
df_ranking = (
    df_corr
    .sort_values("|Pearson r|", ascending=False)
    .reset_index(drop=True)
)

print("=== Ranking completo ===")
display(
    df_ranking[[
        "Tipo Relatório", "Modelo",
        "Pearson r", "Pearson p",
        "Spearman ρ", "|Pearson r|", "N observações"
    ]].round(4)
)

print("\n=== Melhor modelo por tipo de relatório ===")
display(
    df_ranking.groupby("Tipo Relatório").first().reset_index()
    [["Tipo Relatório", "Modelo", "Pearson r", "|Pearson r|"]]
)

=== Ranking completo ===


,Tipo Relatório,Modelo,Pearson r,Pearson p,Spearman ρ,|Pearson r|,N observações
0,copom,NLTK/VADER,0.4292,0.0001,0.4377,0.4292,75
1,copom,Mistral,-0.4218,0.0002,-0.4099,0.4218,75
2,copom,Média dos Modelos,-0.3029,0.0083,-0.2742,0.3029,75
3,estatisticas,FinBERT-PT-BR,-0.2733,0.0177,-0.3201,0.2733,75
4,estatisticas,TextBlob,0.1739,0.1356,0.1374,0.1739,75
5,copom,BERT Multilingual,0.1706,0.1433,0.1776,0.1706,75
6,estatisticas,Média dos Modelos,-0.0873,0.4563,-0.1425,0.0873,75
7,copom,FinBERT-PT-BR,0.0627,0.5928,0.0502,0.0627,75
8,copom,TextBlob,-0.0604,0.6067,0.1388,0.0604,75
9,estatisticas,Mistral,-0.0438,0.7090,-0.0425,0.0438,75



=== Melhor modelo por tipo de relatório ===


,Tipo Relatório,Modelo,Pearson r,|Pearson r|
0,copom,NLTK/VADER,0.4292,0.4292
1,estatisticas,FinBERT-PT-BR,-0.2733,0.2733


## 8. Seleção final dos modelos para o Notebook 06

Com base no ranking da seção 7, selecionamos os quatro modelos das **Atas do
Copom** com maior correlação absoluta. Os Relatórios de Estatísticas não
aparecem entre os melhores colocados e são excluídos da seleção.

Para cada modelo selecionado, o Notebook 06 receberá as colunas com lags de
1 a 6 meses (construídas na seção 5), e o número ideal de lags será determinado
por RMSE, seguindo a mesma lógica adotada para a inadimplência no Notebook 02.

In [62]:
N_SELECIONADOS = 4

df_selecionados = (
    df_ranking
    .head(N_SELECIONADOS)
    .copy()
)

print("=== Modelos selecionados para o Notebook 06 ===")
display(
    df_selecionados[[
        "Tipo Relatório", "Modelo",
        "Pearson r", "|Pearson r|"
    ]].reset_index(drop=True)
)

print("\nPrefixos das colunas a construir no Notebook 06:")
for _, row in df_selecionados.iterrows():
    base = f"copom_{row['col']}"
    print(f"  {base}_L1 ... {base}_L6")

=== Modelos selecionados para o Notebook 06 ===


,Tipo Relatório,Modelo,Pearson r,|Pearson r|
0,copom,NLTK/VADER,0.4292,0.4292
1,copom,Mistral,-0.4218,0.4218
2,copom,Média dos Modelos,-0.3029,0.3029
3,estatisticas,FinBERT-PT-BR,-0.2733,0.2733



Prefixos das colunas a construir no Notebook 06:
  copom_score_nltk_L1 ... copom_score_nltk_L6
  copom_score_mistral_L1 ... copom_score_mistral_L6
  copom_score_media_L1 ... copom_score_media_L6
  copom_score_finbert_L1 ... copom_score_finbert_L6


Estes são os quatro modelos de sentimento selecionados para o Notebook 06.
A contribuição de cada um para a previsão da inadimplência será avaliada
comparando o desempenho do modelo com e sem sentimento, usando RMSE como
critério principal.